# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [3]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except requests.exceptions.RequestException:
            if attempt == tries:
                raise
            time.sleep(attempt)
            continue

        if r.status_code == 200:
            return pd.DataFrame(r.json()['daily'])

        if r.status_code in RETRY_STATUSES:
            if attempt == tries:
                raise
            time.sleep(attempt)
            continue

        raise RuntimeError(f'weather API error {r.status_code}')

    raise RuntimeError('all attempts failed')


    pass

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [4]:
used_fallback = False

try:
    df = get_weather("2026-09-01", "2026-10-05")
    df = df[df['time'].isin(["2026-09-05", "2026-10-03"])]

except Exception:
    used_fallback = True
    df = sample_weather.copy()


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [5]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx = df.copy()
    wx['source'] = 'open-meteo'


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [6]:
wx = wx.rename(columns={
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

wx['date'] = pd.to_datetime(wx['time'])
wx = wx[['date', 'temp_max', 'precip_mm', 'source']]


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [7]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'

print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [8]:
wetter = wx.sort_values('precip_mm', ascending=False).iloc[0]
drier = wx.sort_values('precip_mm', ascending=True).iloc[0]

diff = wetter['precip_mm'] - drier['precip_mm']

print(f"{wetter['date'].date()} was wetter than {drier['date'].date()} by {diff:.1f} mm of precipitation.")


2026-10-03 was wetter than 2026-09-05 by 12.2 mm of precipitation.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [10]:
print("Data source:")
print(f"  endpoint: {BASE}")
print(f"  coordinates: lat=38.03, lon=-78.51")
print(f"  date range: 2026-09-01 to 2026-10-05")
print(f"  pulled at: {pd.Timestamp.now()}")
print(f"  data origin: {'fallback sample' if used_fallback else 'open-meteo'}")


Data source:
  endpoint: https://archive-api.open-meteo.com/v1/archive
  coordinates: lat=38.03, lon=-78.51
  date range: 2026-09-01 to 2026-10-05
  pulled at: 2026-10-11 22:53:57.816740
  data origin: open-meteo


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather quietly returned an empty DataFrame, the rest of the pipeline would march forward as if nothing were wrong. The merge onto 200 days of sales would still succeed, because panadas happily merge an empty table without complaining. That leaves you with a full sales table but no weather values, so every precipiation entry ends up missing or gets filled with zero. When you chart rain against revenue, the rainfall line looks perfectly flat (basicially "it never rained"). Nothing in the chart looks obviosuly broken, and because the pipeline didn't raise an error, nobody has a reason to suspect the weather data failed. The analysis looks clean, the plot renders normally, and the mistake hides inside the data insteread of showing up in the console.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [12]:
# Charlottesville
try:
    df_cv = get_weather("2026-09-01", "2026-10-05", lat=38.03, lon=-78.51)
    df_cv = df_cv[df_cv['time'].isin(["2026-09-05", "2026-10-03"])]
    cv_fallback = False
except Exception:
    cv_fallback = True
    df_cv = FALLBACK.copy()

df_cv = df_cv.rename(columns={
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})
df_cv['date'] = pd.to_datetime(df_cv['time'])
df_cv['source'] = 'fallback sample' if cv_fallback else 'open-meteo'
df_cv['city'] = 'Charlottesville'
df_cv = df_cv[['date', 'temp_max', 'precip_mm', 'source', 'city']]


# Orlando
try:
    df_or = get_weather("2026-09-01", "2026-10-05", lat=28.54, lon=-81.38)
    df_or = df_or[df_or['time'].isin(["2026-09-05", "2026-10-03"])]
    or_fallback = False
except Exception:
    or_fallback = True
    df_or = FALLBACK.copy()

df_or = df_or.rename(columns={
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})
df_or['date'] = pd.to_datetime(df_or['time'])
df_or['source'] = 'fallback sample' if or_fallback else 'open-meteo'
df_or['city'] = 'Orlando'
df_or = df_or[['date', 'temp_max', 'precip_mm', 'source', 'city']]

wx_long = pd.concat([df_cv, df_or], ignore_index=True)


In [13]:
wx_long

,date,temp_max,precip_mm,source,city
0,2026-09-05,29.4,2.5,open-meteo,Charlottesville
1,2026-10-03,22.0,14.7,open-meteo,Charlottesville
2,2026-09-05,32.1,3.9,open-meteo,Orlando
3,2026-10-03,30.4,14.5,open-meteo,Orlando
